# Dark.Tyro M4 — two proof-of-concept demos

**FIT5230 Malicious AI · Milestone 4 (individual) · Wanting Zhao (35507071)**

This notebook backs two sections of my M4 report. It reuses the M3 pipeline unchanged (same ten
Helen faces, same PhotoGuard shield, same Stable Diffusion inpainting editor, same two scores) and
adds two new pieces.

| demo | report section | the question |
|---|---|---|
| **Part A · masked diffusion purification (D)** | Unconstrained enhancement | Can a few denoising steps of a diffusion model replace IMPRESS's 100-iteration optimisation loop, inside the same mask? |
| **Part B · washed-photo detector** | Role reversal (attacker → defender) | If I defended PhotoGuard, could I tell that a photo has been washed? |

## The pipeline (M3, plus one new wash)

```
clean ─PhotoGuard─▶ protected ─wash─▶ washed ─editor─▶ edit      (compared with the edit of the CLEAN photo)
                                  ├─ A  IMPRESS, 100 optimisation iterations          (M3)
                                  ├─ C  A + our mask: wash only the region the editor keeps   (M2/M3)
                                  └─ D  NEW: add noise, denoise a few steps with SD, keep it only inside the mask
```

## Two scores (exactly as in M3)

- **Photo damage** = LPIPS(washed photo, clean photo). Lower is better.
- **Recovery** = `R_pipe = (SSIM_washed − SSIM_protected) / (1 − SSIM_protected)`; each SSIM compares an
  edit with the edit of the clean photo. Higher is better. Run-to-run noise: **±3 percentage points**.
- New for Part A: **wash time per face** (seconds), because D's claim is about cost.

## Stopping rules (written before running)

- **Part A.** Two strengths (0.1, 0.2), one session. **Success** = D's photo damage ≤ C's, with recovery
  within 3 pp of C, at lower wash time than A. **If it fails**, the measured table is the result and
  the report says so.
- **Part B.** One feature, threshold fixed on **40 held-out clean faces** (never the ten test faces).
  **Success** = at a 5 % false-alarm rate, at least 8 of 10 photos flagged for at least one wash.
  **If it fails**, the overlap plot is the result.

## How to run

**Runtime → Change runtime type → T4 GPU**, then **Run all**. About **80 min** from scratch.
To skip the ~60 min of M3 work, give cell 1 the M3 `part1_bundle.zip` (`PART1_FROM`); the notebook
then reuses M3's protected photos and versions N, A, C and only runs D and the detector (~20 min).
Set `QUICK_TEST = True` first for a 15-minute plumbing check (numbers meaningless).

In [ ]:
# 1 · SETTINGS — the only cell you edit.
QUICK_TEST  = False   # True = 2 faces, tiny iteration counts. Plumbing check only.
N_IMAGES    = 10      # the ten pinned M3 test faces
N_CALIB     = 40      # held-out clean Helen faces, used ONLY to set the detector threshold (Part B)

PARAMS = dict(attack_type='l2', pg_eps=16, pg_step_size=1, pg_eta=1, diff_steps=4, guidance=7.5, seed=0,
              pur_eps=0.1, pur_lr=0.005, pur_alpha=0.01, pur_noise=0.05,          # IMPRESS wash (A)
              prompt='a person in an airplane', test_guidance=7.5, test_diff_steps=50)   # the editor
PG_ITERS, PG_REPS = 40, 2          # the shield: identical to M2/M3

D_STRENGTHS = [0.1, 0.2]           # Part A: fraction of the noise schedule. 0.1 of 50 steps = 5 denoising steps
D_STEPS     = 50
D_MODEL     = 'stable-diffusion-v1-5/stable-diffusion-v1-5'   # the base model the inpainting editor was built from

VERSIONS = [dict(name='N_no_wash',     iters=0,   masked=False),
            dict(name='A_impress_100', iters=100, masked=False),
            dict(name='C_tyro_masked', iters=100, masked=True, reuse='A_impress_100')] + \
           [dict(name=f'D_diffusion_{int(s * 100):02d}', iters=0, masked=True, strength=s) for s in D_STRENGTHS]

PART1_FROM = ''   # optional: path to M3's part1_bundle.zip (or its unzipped folder) to reuse N, A, C

In [ ]:
# 2 · Platform, GPU check, and where results are saved (as in M3).
import os, sys, re, json, time, glob, shutil, zipfile, pathlib, subprocess, threading
import numpy as np, pandas as pd, torch
from PIL import Image, ImageFilter

BASE, PLATFORM = ((pathlib.Path('/kaggle/working'), 'kaggle') if os.path.isdir('/kaggle/working')
                  else (pathlib.Path('/content'), 'colab') if os.path.isdir('/content')
                  else (pathlib.Path.cwd(), 'local'))
DEVICE      = 'cuda:0' if torch.cuda.is_available() else 'cpu'
IMPRESS_DIR = BASE / 'Impress'
ROOT        = BASE / 'helen_face'      # IMPRESS's scripts read ../helen_face
CALIB       = BASE / 'calib_clean'     # held-out clean faces for Part B, kept OUTSIDE helen_face
SUBENV      = dict(os.environ, PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')

GPU_OK = torch.cuda.is_available()
if GPU_OK:
    _tag   = 'sm_%d%d' % torch.cuda.get_device_capability(0)
    GPU_OK = _tag in torch.cuda.get_arch_list()
    print(f'gpu       : {torch.cuda.get_device_name(0)} ({_tag})' + ('' if GPU_OK else '  !! no kernels: switch to a T4'))
else:
    print('!! no GPU. Colab: Runtime -> Change runtime type -> T4.  Kaggle: Accelerator -> GPU T4')

if QUICK_TEST:
    N_IMAGES, N_CALIB, PG_ITERS = 2, 6, 10
    for _v in VERSIONS: _v['iters'] = min(_v['iters'], 10)
    print('*** QUICK TEST: plumbing only, the numbers mean nothing ***')

RUN_TAG = ('TEST_' if QUICK_TEST else '') + 'm4_demos'
OUT     = BASE / 'tyro_results' / RUN_TAG
OUT.mkdir(parents=True, exist_ok=True)

DRIVE_DIR = None
if PLATFORM == 'colab':
    try:
        from google.colab import drive; drive.mount('/content/drive')
        DRIVE_DIR = pathlib.Path('/content/drive/MyDrive/FIT5230_M4') / RUN_TAG
        DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    except Exception as e:
        print(f'!! Drive not mounted ({e}). Download {OUT} before closing the tab.')
print(f'platform  : {PLATFORM} · {DEVICE} · results -> {OUT}' + (f' (copied to {DRIVE_DIR})' if DRIVE_DIR else ''))

In [ ]:
# 3 · Install, clone IMPRESS, apply the two 2026 repairs (same as M3; no training logger needed here).
import importlib.util
_need    = {'lpips': 'lpips', 'gdown': 'gdown', 'ftfy': 'ftfy', 'diffusers': 'diffusers'}
_missing = [pkg for mod, pkg in _need.items() if importlib.util.find_spec(mod) is None]
if _missing:   # --no-deps: IMPRESS's unpinned requirements would pull a whole new torch stack
    subprocess.run(f'pip install -q --no-deps {" ".join(_missing)}', shell=True, check=False)
print('installed:', _missing or 'nothing needed')

if not IMPRESS_DIR.exists():
    subprocess.run(f'git clone -q --depth 1 https://github.com/AAAAAAsuka/Impress.git {IMPRESS_DIR}',
                   shell=True, check=True)
for p in IMPRESS_DIR.glob('*.py'):
    s = p.read_text()
    t = s.replace('runwayml/stable-diffusion-inpainting', 'stable-diffusion-v1-5/stable-diffusion-inpainting')
    t = re.sub(r'^[ \t]*revision="fp16",[ \t]*\r?\n', '', t, flags=re.M)
    if t != s: p.write_text(t)
print('IMPRESS cloned + repaired')

In [ ]:
# 4 · Data: the ten PINNED M3 test faces, plus N_CALIB OTHER clean faces set aside for Part B.
import gdown
PINNED = sorted(['1233476865_1.png', '1525918600_1.png', '178046512_1.png', '181707205_1.png',
                 '1961032923_1.png', '2061993362_1.png', '2099073485_1.png', '2139626906_1.png',
                 '2167874246_1.png', '221629697_1.png'])[:N_IMAGES]
_keep = set(PINNED)
_zip  = BASE / 'helen_face_dataset.zip'
if not _zip.exists():
    gdown.download(id='16xISe7M_DlSqM2Zf2lWI4JJXcdsDEPsl', output=str(_zip), quiet=True)

def _extract():
    ROOT.mkdir(exist_ok=True)
    with zipfile.ZipFile(_zip) as z: z.extractall(ROOT)

_calib_ok = CALIB.is_dir() and len(list(CALIB.glob('*.png'))) >= N_CALIB
if not (ROOT / 'clean').exists() or not _keep <= set(os.listdir(ROOT / 'clean')) or not _calib_ok:
    _extract()
if not _calib_ok:   # held-out faces: the first N_CALIB clean faces that are NOT test faces
    CALIB.mkdir(exist_ok=True)
    others = sorted(set(os.listdir(ROOT / 'clean')) - _keep)[:N_CALIB]
    for n in others:
        Image.open(ROOT / 'clean' / n).convert('RGB').resize((512, 512)).save(CALIB / n)
for _sub in ('clean', 'mask'):          # IMPRESS processes every file in these folders
    for _n in os.listdir(ROOT / _sub):
        if _n not in _keep: os.remove(ROOT / _sub / _n)
assert set(os.listdir(ROOT / 'clean')) == _keep, 'test faces not in place'
assert not (set(os.listdir(CALIB)) & _keep), 'LEAKAGE: a test face is in the calibration set'
print(f'{len(_keep)} test faces · {len(os.listdir(CALIB))} held-out calibration faces (no overlap)')

## The pipeline

Stages 1–3 are M3's code, unchanged. The only new function is `purify_diffusion` (version D).

**How D works, in three lines.** A diffusion model is trained to remove Gaussian noise. So:
(1) encode the protected photo and add a *small* amount of Gaussian noise (strength 0.1 = the
first 10 % of the noise schedule); (2) let Stable Diffusion denoise it for the matching few steps,
with an empty prompt; (3) keep the result only inside the mask, as in C. The shield is a small,
structured signal: the added noise drowns it and the denoiser does not know how to put it back.
This is the idea of DiffPure (Nie et al., ICML 2022) and AudioPure (Wu et al., ICLR 2023).
**No optimisation loop**, so it needs no gradients and no per-image training.

In [ ]:
# 5 · The pipeline. Nothing runs until cell 6.
from skimage.metrics import structural_similarity as _ssim
P = dict(PARAMS, pur_iters=0)
TIMES = {}    # version -> wash seconds per face

def adv_dir():      return (f"{ROOT}/adv_{P['attack_type']}_eps{P['pg_eps']}_step{P['pg_step_size']}_iter{PG_ITERS}"
                            f"grad_reps{PG_REPS}_eta{P['pg_eta']}_diff_steps{P['diff_steps']}_guidance{P['guidance']}_seed{P['seed']}")
def pur_dir():      return (f"{ROOT}/pur_eps{P['pur_eps']}_pur_iters{P['pur_iters']}_pur_lr{P['pur_lr']}"
                            f"_pur_alpha{P['pur_alpha']}_pur_noise{P['pur_noise']}/")
def adv_diff_dir(): return re.sub('adv', 'adv_diff', adv_dir())
def pur_diff_dir(): return re.sub('pur', 'pur_diff', pur_dir())

SHIELD = (f"--attack_type={P['attack_type']} --pg_eps={P['pg_eps']} --pg_step_size={P['pg_step_size']} "
          f"--pg_eta={P['pg_eta']} --pg_iters={PG_ITERS} --pg_grad_reps={PG_REPS} --diff_steps={P['diff_steps']} "
          f"--guidance={P['guidance']} --parallel_index=-1 --device={DEVICE}")
def wash_args():
    return (f"--pur_eps={P['pur_eps']} --pur_iters={P['pur_iters']} --pur_lr={P['pur_lr']} "
            f"--pur_alpha={P['pur_alpha']} --pur_noise={P['pur_noise']}")

def sh(script, extra=''):
    """Run one IMPRESS script; returns its wall time in seconds."""
    print(f'  $ {script} ...', flush=True)
    t0, stop = time.time(), threading.Event()
    def _beat():
        while not stop.wait(300): print(f'    ... {(time.time() - t0) / 60:.0f} min', flush=True)
    threading.Thread(target=_beat, daemon=True).start()
    try:
        r = subprocess.run(f'{sys.executable} -u {script} {SHIELD} {extra}', shell=True, cwd=str(IMPRESS_DIR),
                           env=SUBENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    finally:
        stop.set()
    if r.returncode != 0:
        print(r.stdout[-3000:]); raise RuntimeError(f'{script} failed (exit {r.returncode})')
    dt = time.time() - t0
    print(f'    done in {dt / 60:.1f} min', flush=True)
    return dt

def copy_pngs(src, dst):
    os.makedirs(dst, exist_ok=True)
    for f in sorted(glob.glob(f'{src}/*.png')): shutil.copy(f, dst)

def bridge():
    """IMPRESS's protect script writes adv_..., its wash script reads adapt_adv_... (known repo bug)."""
    src = adv_dir(); dst = src.replace('/adv_', '/adapt_adv_')
    assert len(glob.glob(f'{src}/*.png')) == N_IMAGES, f'expected {N_IMAGES} protected photos in {src}'
    shutil.rmtree(dst, ignore_errors=True); shutil.copytree(src, dst)

def restrict_to_mask(path, feather=3):
    """OUR M2 METHOD. Keep the wash inside the kept region, put the protected pixels back elsewhere."""
    n    = os.path.basename(path)
    pur  = np.asarray(Image.open(path).convert('RGB'), np.float32)
    prot = np.asarray(Image.open(f'{adv_dir()}/{n}').convert('RGB').resize(pur.shape[1::-1]), np.float32)
    m    = Image.open(f'{ROOT}/mask/{n}').convert('L').resize(pur.shape[1::-1]).filter(ImageFilter.GaussianBlur(feather))
    m    = np.asarray(m, np.float32)[..., None] / 255.0
    Image.fromarray(np.clip(m * pur + (1 - m) * prot, 0, 255).astype(np.uint8)).save(path)

# ---- NEW IN M4 · version D ----------------------------------------------------------------
_I2I = None
def img2img():
    global _I2I
    if _I2I is None:
        from diffusers import StableDiffusionImg2ImgPipeline
        kw = dict(torch_dtype=torch.float16, safety_checker=None, requires_safety_checker=False)
        try:    _I2I = StableDiffusionImg2ImgPipeline.from_pretrained(D_MODEL, variant='fp16', **kw)
        except Exception: _I2I = StableDiffusionImg2ImgPipeline.from_pretrained(D_MODEL, **kw)
        _I2I = _I2I.to(DEVICE); _I2I.set_progress_bar_config(disable=True)
    return _I2I

def purify_diffusion(src, dst, strength):
    """Add noise up to `strength` of the schedule, denoise with an empty prompt. Same seed as the rest."""
    pipe = img2img(); os.makedirs(dst, exist_ok=True)
    files = sorted(glob.glob(f'{src}/*.png'))
    if DEVICE.startswith('cuda'): torch.cuda.synchronize()
    t0 = time.time()
    for f in files:
        x = Image.open(f).convert('RGB')
        y = pipe(prompt='', image=x.resize((512, 512)), strength=strength, num_inference_steps=D_STEPS,
                 guidance_scale=1.0, generator=torch.Generator(DEVICE).manual_seed(P['seed'])).images[0]
        y.resize(x.size).save(f'{dst}/{os.path.basename(f)}')
    if DEVICE.startswith('cuda'): torch.cuda.synchronize()
    return (time.time() - t0) / len(files)

# ---- stages ----------------------------------------------------------------------------------
def protect():
    print('=' * 72 + f'\nSTAGE 1 · PhotoGuard protects {N_IMAGES} faces (once, shared by every version)')
    sh('pg_mask_diff_helen.py'); bridge()

def edit():
    shutil.rmtree(pur_diff_dir(), ignore_errors=True)
    sh('pg_generate.py', f"{wash_args()} --test_guidance={P['test_guidance']} "
                         f"--test_diff_steps={P['test_diff_steps']} --prompt=\"{P['prompt']}\"")

def save(v):
    dest = OUT / v['name']; shutil.rmtree(dest, ignore_errors=True); (dest / 'clean512').mkdir(parents=True)
    for f in sorted(glob.glob(f'{ROOT}/clean/*.png')):
        Image.open(f).convert('RGB').resize((512, 512)).save(dest / 'clean512' / os.path.basename(f))
    for sub, src in [('protected', adv_dir()), ('washed', pur_dir()), ('edit_clean', f'{ROOT}/clean_diff'),
                     ('edit_protected', adv_diff_dir()), ('edit_washed', pur_diff_dir())]:
        copy_pngs(src, dest / sub)
    (dest / 'params.json').write_text(json.dumps(dict(P, version=v['name'], strength=v.get('strength'),
        masked=v['masked'], pg_iters=PG_ITERS, pg_grad_reps=PG_REPS, platform=PLATFORM,
        saved=time.strftime('%Y-%m-%d %H:%M')), indent=2))
    print(f"  [saved] {v['name']} -> {dest}", flush=True)

def run_version(v):
    print('=' * 72 + f"\n{v['name']}")
    P['pur_iters'] = v['iters']
    for d in (pur_dir(), pur_diff_dir()): shutil.rmtree(d, ignore_errors=True)
    if v.get('strength'):                                # D: diffusion purification
        TIMES[v['name']] = purify_diffusion(adv_dir(), pur_dir(), v['strength'])
    elif v.get('reuse'):                                 # C: A's washed photo + the mask
        copy_pngs(OUT / v['reuse'] / 'washed', pur_dir()); TIMES[v['name']] = TIMES.get(v['reuse'])
    elif v['iters'] == 0:                                # N: no wash
        copy_pngs(adv_dir(), pur_dir()); TIMES[v['name']] = 0.0
    else:                                                # A: IMPRESS optimisation loop
        TIMES[v['name']] = sh('pg_mask_pur_helen.py', wash_args()) / N_IMAGES
    if v['masked']:
        for f in sorted(glob.glob(f'{pur_dir()}/*.png')): restrict_to_mask(f)
    if TIMES.get(v['name']) is not None: print(f"  wash time: {TIMES[v['name']]:.1f} s per face")
    print('  edit'); edit(); save(v)

def load_m3(src):
    """Reuse M3 Part 1's versions N, A, C and its protected photos (identical shield)."""
    src = pathlib.Path(src); print(f'reusing M3 results from {src}')
    if src.suffix == '.zip':
        with zipfile.ZipFile(src) as z:
            z.extractall(OUT, members=[m for m in z.namelist() if m.split('/')[0] in
                                       ('N_no_wash', 'A_impress_100', 'C_tyro_masked')])
    else:
        for d in ('N_no_wash', 'A_impress_100', 'C_tyro_masked'):
            shutil.copytree(src / d, OUT / d, dirs_exist_ok=True)
    shutil.rmtree(adv_dir(), ignore_errors=True)
    copy_pngs(OUT / 'N_no_wash' / 'protected', adv_dir()); bridge()

def ssim_pair(pa, pb):
    a, b = (np.asarray(Image.open(p).convert('RGB'), np.uint8) for p in (pa, pb))
    if a.shape != b.shape:
        h, w = min(a.shape[0], b.shape[0]), min(a.shape[1], b.shape[1])
        a, b = (np.asarray(Image.fromarray(x).resize((w, h))) for x in (a, b))
    return float(_ssim(a, b, channel_axis=2, data_range=255))
print('pipeline ready')

In [ ]:
# 6 · RUN Part A. Each version is saved the moment it finishes.
assert GPU_OK, 'no usable GPU: see cell 2'
T0 = time.time()
if PART1_FROM:
    load_m3(PART1_FROM)
    todo = [v for v in VERSIONS if v.get('strength')]
    print('wash time for A is not re-measured when M3 results are reused')
else:
    protect(); todo = VERSIONS
for v in todo: run_version(v)
print(f'\nPart A finished in {(time.time() - T0) / 60:.0f} min:', sorted(d.name for d in OUT.iterdir() if d.is_dir()))

# Part A result · does diffusion purification beat the optimisation loop?

Every version faced the **same protected photos** and the same editor, seed, prompt and mask.
The reference edits (clean, protected) come from N, as in M3.

In [ ]:
# 7 · Score every version on the M3 axes, plus wash time.
import lpips
_net = lpips.LPIPS(net='alex', verbose=False).to(DEVICE).eval()
def _t(q): return (torch.from_numpy(np.asarray(Image.open(q).convert('RGB').resize((512, 512)), np.float32) / 127.5 - 1)
                   .permute(2, 0, 1)[None].to(DEVICE))
def lp(pa, pb):
    with torch.no_grad(): return float(_net(_t(pa), _t(pb)))

REF   = OUT / 'N_no_wash'
FACES = sorted(os.listdir(REF / 'edit_clean'))
ORDER = [v['name'] for v in VERSIONS if (OUT / v['name'] / 'edit_washed').is_dir()]
LABEL = {'N_no_wash': 'N · no wash', 'A_impress_100': 'A · IMPRESS 100 it.', 'C_tyro_masked': 'C · A + mask',
         **{f'D_diffusion_{int(s * 100):02d}': f'D · diffusion {s} + mask' for s in D_STRENGTHS}}

rows = []
for name in ORDER:
    for f in FACES:
        adv = ssim_pair(REF / 'edit_protected' / f, REF / 'edit_clean' / f)
        pur = ssim_pair(OUT / name / 'edit_washed' / f, REF / 'edit_clean' / f)
        rows.append(dict(version=name, face=f, ssim_adv=adv, ssim_pur=pur, R_pipe=100 * (pur - adv) / (1 - adv),
                         damage=lp(OUT / name / 'washed' / f, REF / 'clean512' / f)))
per = pd.DataFrame(rows)

def _summ(g):
    a, p = g.ssim_adv.mean(), g.ssim_pur.mean()
    return pd.Series(dict(photo_damage=g.damage.mean(), recovery=100 * (p - a) / (1 - a),
                          faces_up=int((g.R_pipe > 3).sum()), faces_down=int((g.R_pipe < -3).sum()),
                          wash_s_per_face=TIMES.get(g.name)))
S = per.groupby('version', sort=False).apply(_summ).loc[ORDER].apply(pd.to_numeric)   # None (time unknown) -> NaN
show = S.rename(index=LABEL).rename(columns={'photo_damage': 'photo damage (LPIPS) ↓', 'recovery': 'recovery R_pipe % ↑',
        'faces_up': 'faces > +3 pp', 'faces_down': 'faces < −3 pp', 'wash_s_per_face': 'wash s / face ↓'})
display(show.round(3))

In [ ]:
# 8 · Two single-axis panels (no dual axis): photo damage and recovery for each version.
import matplotlib.pyplot as plt
OURS, BASELINE, NEUTRAL = '#1f6fd1', '#6b6b6b', '#bdbdbd'
col = [NEUTRAL if n.startswith('N') else OURS if n.startswith('D') else '#8fb6e8' if n.startswith('C') else BASELINE
       for n in ORDER]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, key, title in ((axes[0], 'photo_damage', 'Photo damage · LPIPS vs clean (lower is better)'),
                       (axes[1], 'recovery',     'Recovery · R_pipe % (higher is better; grey = ±3 pp noise)')):
    ax.barh([LABEL[n] for n in ORDER][::-1], S[key].values[::-1], color=col[::-1], height=.6)
    for i, v in enumerate(S[key].values[::-1]):
        ax.text(v, i, f' {v:.3f}' if key == 'photo_damage' else f' {v:+.1f}', va='center', fontsize=8, color='#333')
    ax.set_title(title, fontsize=10); ax.grid(axis='x', alpha=.3); ax.spines[['top', 'right']].set_visible(False)
axes[1].axvspan(-3, 3, color='grey', alpha=.15, lw=0)   # the measured ±3 pp run-to-run noise
fig.suptitle(f'Part A · {len(FACES)} faces, identical shield and editor for every version', fontsize=11)
fig.tight_layout(); fig.savefig(OUT / 'figA_damage_recovery.png', dpi=130, bbox_inches='tight'); plt.show()

# Three faces, every version's edit, so the numbers can be checked by eye.
show_faces = FACES[:3]
cols = [('clean photo', REF, 'clean512'), ('TARGET · edit of clean', REF, 'edit_clean'),
        ('edit of protected (N)', REF, 'edit_protected')] + \
       [(LABEL[n], OUT / n, 'edit_washed') for n in ORDER if n != 'N_no_wash']
fig, axes = plt.subplots(len(show_faces), len(cols), figsize=(2.2 * len(cols), 2.3 * len(show_faces)))
axes = np.atleast_2d(axes)
for r, f in enumerate(show_faces):
    for c, (t, d, sub) in enumerate(cols):
        ax = axes[r, c]; ax.imshow(Image.open(d / sub / f)); ax.set_xticks([]); ax.set_yticks([])
        if r == 0: ax.set_title(t, fontsize=8)
fig.tight_layout(); fig.savefig(OUT / 'figA_examples.png', dpi=100, bbox_inches='tight'); plt.show()

In [ ]:
# 9 · Verdict against the stopping rule (written before the run), produced by the data.
def _in(x): return 'inside' if abs(x) <= 3 else 'OUTSIDE'
C_ = S.loc['C_tyro_masked']
for n in [n for n in ORDER if n.startswith('D_')]:
    D_ = S.loc[n]
    dmg_ok = D_.photo_damage <= C_.photo_damage
    rec_ok = D_.recovery >= C_.recovery - 3
    t_A    = TIMES.get('A_impress_100'); t_ok = (t_A is not None) and D_.wash_s_per_face < t_A
    print(f'{LABEL[n]}: damage {D_.photo_damage:.3f} vs C {C_.photo_damage:.3f} ({(D_.photo_damage / C_.photo_damage - 1):+.0%}) · '
          f'recovery {D_.recovery - C_.recovery:+.1f} pp vs C ({_in(D_.recovery - C_.recovery)} the noise) · '
          f'time {D_.wash_s_per_face:.1f} s vs A ' + (f'{t_A:.1f} s' if t_A else 'n/a'))
    print('   ->', 'PASS' if (dmg_ok and rec_ok and (t_ok or t_A is None)) else 'FAIL',
          '' if t_A is not None else '(time check skipped: A was reused from M3)')

# Part B · Role reversal: can a defender tell that a photo was washed?

**The idea comes straight from how the attack works.** IMPRESS's wash *minimises one number*: how
much the photo changes on a round trip through Stable Diffusion's encoder–decoder (VAE). A natural
photo changes by some typical amount. A protected photo changes *more* (that is what IMPRESS looks
for). A photo the wash has optimised, or one that a diffusion model has just regenerated (D), may
change *less than any natural photo*: the attacker leaves the fingerprint of its own objective.

**The feature.** The VAE round-trip error, averaged in 32×32 patches, then the **10th-percentile
patch** = "the most VAE-consistent part of the photo". Taking the low tail rather than the mean is
what lets the detector see the masked washes (C, D), where only the kept region was washed.

**No leakage.** The threshold is set on **40 held-out clean faces** that never appear in the test.
It is the value 95 % of those clean faces stay above, i.e. a 5 % false-alarm rate.

In [ ]:
# 10 · The detector: VAE round-trip error, low-tail patch statistic.
from diffusers import AutoencoderKL
from sklearn.metrics import roc_auc_score
vae = AutoencoderKL.from_pretrained(D_MODEL, subfolder='vae', torch_dtype=torch.float32).to(DEVICE).eval()

@torch.no_grad()
def vae_score(path, patch=32, q=10):
    x = _t(path).float()                                       # [-1, 1], 512x512
    y = vae.decode(vae.encode(x).latent_dist.mean).sample.clamp(-1, 1)
    err = ((x - y) ** 2).mean(1, keepdim=True)                 # per-pixel round-trip error
    pat = torch.nn.functional.avg_pool2d(err, patch).flatten() # one value per 32x32 patch
    return float(np.percentile(pat.cpu().numpy(), q) * 1e3)    # x1000 for readability

GROUPS = {'clean (held-out, calibration)': sorted(CALIB.glob('*.png')),
          'clean (test)':                  sorted((REF / 'clean512').glob('*.png')),
          'protected (N)':                 sorted((REF / 'protected').glob('*.png'))}
GROUP_VER = {}
for n in ORDER[1:]:
    GROUPS[f'washed · {LABEL[n]}'] = sorted((OUT / n / 'washed').glob('*.png')); GROUP_VER[f'washed · {LABEL[n]}'] = n
scores = pd.DataFrame([dict(group=g, file=p.name, score=vae_score(p)) for g, ps in GROUPS.items() for p in ps])

THR = np.percentile(scores[scores.group == 'clean (held-out, calibration)'].score, 5)
scores['flagged_washed'] = scores.score < THR
tab = scores.groupby('group', sort=False).agg(n=('score', 'size'), median_score=('score', 'median'),
                                              flagged_as_washed=('flagged_washed', 'sum'))
neg = scores[scores.group.str.startswith('clean')].score
tab['AUC vs all clean'] = [np.nan if g.startswith('clean') else
                           roc_auc_score([0] * len(neg) + [1] * (scores.group == g).sum(),
                                         list(-neg) + list(-scores[scores.group == g].score))
                           for g in tab.index]
print(f'threshold = {THR:.3f} (5th percentile of {len(GROUPS["clean (held-out, calibration)"])} held-out clean faces)')
display(tab.round(3))

In [ ]:
# 11 · One axis: every photo's score, grouped, with the threshold.
fig, ax = plt.subplots(figsize=(9, 0.5 * len(GROUPS) + 1.2))
names = list(GROUPS)[::-1]
rng = np.random.default_rng(0)
for i, g in enumerate(names):
    s = scores[scores.group == g].score.values
    c = NEUTRAL if g.startswith('clean') else '#c98b2b' if g.startswith('protected') else OURS
    ax.scatter(s, i + rng.uniform(-.15, .15, len(s)), s=22, color=c, edgecolor='white', linewidth=.6, zorder=3)
ax.axvline(THR, color='#222', ls='--', lw=1); ax.set_ylim(-1, len(names) - .5)
ax.text(THR, -.75, ' threshold (5 % false alarms)', fontsize=8, va='center')
ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=8); ax.set_xscale('log')
ax.set_xlabel('VAE round-trip error of the most consistent 10 % of the photo (×1000, log scale) · left of the line = flagged as washed')
ax.grid(axis='x', alpha=.3); ax.spines[['top', 'right']].set_visible(False)
ax.set_title('Part B · a washed photo is "too consistent" for Stable Diffusion\'s own VAE', fontsize=10)
fig.tight_layout(); fig.savefig(OUT / 'figB_detector.png', dpi=130, bbox_inches='tight'); plt.show()

ok = [g for g in tab.index if g.startswith('washed') and tab.loc[g, 'flagged_as_washed'] >= 0.8 * tab.loc[g, 'n']]
fp = int(tab.loc['clean (test)', 'flagged_as_washed'])
print(f'clean test faces wrongly flagged: {fp} of {int(tab.loc["clean (test)", "n"])}')
print('stopping rule:', 'PASS for ' + ', '.join(ok) if ok else 'FAIL: no wash flagged on >= 8 of 10 faces')

## The attacker's counter-move (one round further)

A detector invites an answer. The cheapest one: after washing, add a little fresh noise so the
photo looks "naturally inconsistent" again. This cell measures how much noise that takes and what
it costs the photo. It is a limit test of the detector, not part of the stopping rule.

In [ ]:
# 12 · Counter-move: Gaussian noise (sigma in grey levels) added to the washed photos of the best-detected wash.
target = max([g for g in tab.index if g.startswith('washed')], key=lambda g: tab.loc[g, 'flagged_as_washed'])
vname  = GROUP_VER[target]
tmp = OUT / '_countermove'; shutil.rmtree(tmp, ignore_errors=True); tmp.mkdir()
cm = []
for sigma in (0, 1, 2, 4):
    flags, dmg = 0, []
    for f in FACES:
        x = np.asarray(Image.open(OUT / vname / 'washed' / f).convert('RGB').resize((512, 512)), np.float32)
        x = np.clip(x + np.random.default_rng(1).normal(0, sigma, x.shape), 0, 255).astype(np.uint8)
        q = tmp / f; Image.fromarray(x).save(q)
        flags += vae_score(q) < THR; dmg.append(lp(q, REF / 'clean512' / f))
    cm.append(dict(sigma_grey_levels=sigma, flagged=f'{flags} of {len(FACES)}', photo_damage=np.mean(dmg)))
print(f'counter-move against: {target}')
display(pd.DataFrame(cm).round(3))

## Limitations

1. **The shield is M3's weak setting** (40 × 2 iterations, 1/25 of the reference), so recovery has little
   to separate; photo damage and wash time are the reliable axes in Part A.
2. **n = 10 test faces, one prompt, one editor.** Part B's threshold comes from 40 faces, so "5 % false alarms"
   is itself an estimate with wide error bars.
3. **D's wash time excludes the one-off model load** (IMPRESS's includes its own). The per-face cost is what scales.
4. **Part B assumes the defender has the same VAE as the attacker.** For Stable Diffusion 1.x this is public,
   but a different editor would need its own detector.

In [ ]:
# 13 · Collect tables + figures into one small zip (and Drive on Colab).
REPORT = OUT / 'REPORT_m4'
shutil.rmtree(REPORT, ignore_errors=True); REPORT.mkdir()
per.round(5).to_csv(REPORT / 'partA_per_face.csv', index=False)
S.round(5).to_csv(REPORT / 'partA_summary.csv')
scores.round(5).to_csv(REPORT / 'partB_scores.csv', index=False)
tab.round(5).to_csv(REPORT / 'partB_summary.csv')
pd.DataFrame(cm).round(5).to_csv(REPORT / 'partB_countermove.csv', index=False)
for p in OUT.glob('fig*.png'): shutil.copy(p, REPORT / p.name)
zp = shutil.make_archive(str(REPORT), 'zip', root_dir=str(REPORT))
if DRIVE_DIR: shutil.copy(zp, DRIVE_DIR / os.path.basename(zp))
print(os.path.basename(zp), sorted(os.listdir(REPORT)))